### Seasonal Decomposition

In [ ]:
import pandas as pd
import numpy as np
import os, pathlib
from pathlib import Path

In [ ]:
block_df = pd.read_parquet('hhblock_df.parquet')

In [ ]:
exp_block_df = compact_to_expanded(
    block_df, timeseries_col='energy_consumption',
    static_cols=['frequency', 'series_length', 'stdorToU', 'Acorn', 'Acorn_grouped'],
    time_varying_cols=['pressure', 'apparentTemperature', 'windSpeed', 'precipType', 'icon', 'humidity', 'summary'],
    ts_identifier='LCLid')

In [ ]:
exp_block_df.head(1)

Train, Test and Validation Sets

In [ ]:
test_mask = (exp_block_df.timestamp.dt.year==2014) & (exp_block_df.timestamp.dt.month==2)
validation_mask = (exp_block_df.timestamp.dt.year==2014) & (exp_block_df.timestamp.dt.month==1)

train = exp_block_df[~(test_mask | validation_mask)]
validation = exp_block_df[validation_mask]
test = exp_block_df[test_mask]
train.shape, validation.shape, test.shape

((11855088, 15), (595200, 15), (518400, 15))

#### Baseline

In [ ]:
train_df = train[['LCLid', 'timestamp', 'energy_consumption', 'frequency']]
validation_df = validation[
    ['LCLid', 'timestamp', 'energy_consumption', 'frequency']]
test_df = test[['LCLid', 'timestamp', 'energy_consumption', 'frequency']]

In [ ]:
train_df.timestamp.min(), train_df.timestamp.max(), \
test_df.timestamp.min(), test_df.timestamp.max(), \
validation_df.timestamp.min(), validation_df.timestamp.max()

In [ ]:
freq_ = train_df.iloc[0]['frequency']
timeseries_train = train_df.loc[train_df.LCLid=='MAC000322', ['LCLid', 'timestamp', 'energy_consumption']]
timeseries_validation = validation_df.loc[validation_df.LCLid=='MAC000322', ['LCLid', 'timestamp', 'energy_consumption']]
timeseries_test = test_df.loc[test_df.LCLid=='MAC000322', ['LCLid', 'timestamp', 'energy_consumption']]

In [ ]:
ts = SeasonalInterpolation(seasonal_period=48*7).fit_transform(ts_df.energy_consumption.values.reshape(-1,1)).squeeze()

Seasonal Decomposition

In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose

In [ ]:
#Does not support misssing values, so using imputed ts instead
res = seasonal_decompose(ts, period=7*48, model="additive", extrapolate_trend="freq", filt=np.repeat(1/(30*48), 30*48))

In [ ]:
fig = decomposition_plot(ts_df.index, res.observed, res.seasonal, res.trend, res.resid)
fig

In [ ]:
#Let's zoom in on a few days to better see the seasonality extracted
fig.update_xaxes(type="date", range=["2012-11-4", "2012-12-4"])
fig #.show(renderer="svg")

Seasonality and Trend Decomposition using Loess (STL)

In [ ]:
#Supports missing values and expects a series or dataframe with datetime index
stl = STL(seasonality_period=7*48, model = "additive")
res_new = stl.fit(ts_df.energy_consumption)

In [ ]:
fig = res_new.plot(interactive=True)
fig

In [ ]:
#Let's zoom in on a few days to better see the seasonality extracted
fig.update_xaxes(type="date", range=["2012-11-4", "2012-12-4"])
fig

Seasonality and Trend Decomposition using Loess and Fourier Terms (Fourier Decomposition)

In [ ]:
#Doesn't support missing values, and expects a series or datafeame with datetime index
stl = FourierDecomposition(seasonality_period="hour", model = "additive", n_fourier_terms=5)
res_new = stl.fit(pd.Series(ts.squeeze(), index=ts_df.index))

In [ ]:
fig = res_new.plot(interactive=True)
fig

In [ ]:
fig.update_xaxes(type="date", range=["2012-11-4", "2012-12-4"])
fig

Custom Seasonality

In [ ]:
#Making a custom seasonality term
ts_df["dayofweek"] = ts_df.index.dayofweek
ts_df["hour"] = ts_df.index.hour
#Creating a sorted unique combination df
map_df = ts_df[["dayofweek","hour"]].drop_duplicates().sort_values(["dayofweek", "hour"])
# Assigning an ordinal variable to capture the order
map_df["map"] = np.arange(1, len(map_df)+1)
# mapping the oprdinal mapping back to the original df and getting the seasonality array
seasonality = ts_df.merge(map_df, on=["dayofweek","hour"], how='left', validate="many_to_one")['map']

In [ ]:
stl = FourierDecomposition(model = "additive", n_fourier_terms=50)
res_new = stl.fit(pd.Series(ts, index=ts_df.index), seasonality=seasonality)

In [ ]:
fig = res_new.plot(interactive=True)
fig

In [ ]:
fig.update_xaxes(type="date", range=["2012-11-4", "2012-12-4"])
fig

Multiple Seasonality Decomposition using Loess (MSTL)

Using Averages as the seasonal model

In [ ]:
stl = MultiSeasonalDecomposition(seasonal_model="averages",seasonality_periods=[48*7, 48], model = "additive")
res_new = stl.fit(pd.Series(ts, index=ts_df.index))

In [ ]:
fig = res_new.plot(interactive=True)
fig

In [ ]:
fig.update_xaxes(type="date", range=["2012-11-4", "2012-12-4"])
fig

Using Fourier Decomposition as seasonal model

In [ ]:
stl = MultiSeasonalDecomposition(seasonal_model="fourier",seasonality_periods=["day_of_year", "day_of_week", "hour"], model = "additive", n_fourier_terms=10)
res_new = stl.fit(pd.Series(ts, index=ts_df.index))

In [ ]:
fig = res_new.plot(interactive=True)
fig

In [ ]:
fig.update_xaxes(type="date", range=["2012-11-4", "2012-12-4"])
fig

MSTL decomposition in STATSMODELS


break out time series up in to daily and weekly seasonal components.

In [ ]:
exp_block_df.head()

In [ ]:
ts_df_mstl = exp_block_df[exp_block_df.LCLid=="MAC000322"][['timestamp','energy_consumption']].set_index('timestamp')
ts_df_mstl.head()

In [ ]:
stl_kwargs = {"seasonal_deg": 0, }
mstl = MSTL(season_length=[48, 48*7],
            #windows=[101, 101],  # Setting this large along with `seasonal_deg=0` will force the seasonality to be periodic.
            stl_kwargs = stl_kwargs,
            )
mstl_result = mstl.fit(pd.Series(ts, index=ts_df_mstl.index))

In [ ]:
mstl_result.model_['trend'].head()

In [ ]:
print(mstl_result.model_.keys())
mstl_result.model_['seasonal48'].head()
mstl_result.model_['seasonal336'].head()
mstl_result.model_['remainder'].head()

In [ ]:
#res.resid.head()

In [ ]:
#sns.set_style("darkgrid")
plt.rc("figure", figsize=(16, 12))
plt.rc("font", size=13)

# Extracting components from MSTL result
observed_mstl = mstl_result.model_['data']
trend_mstl = mstl_result.model_['trend']
# Summing the two seasonal components
seasonal_mstl = mstl_result.model_['seasonal48'] + mstl_result.model_['seasonal336']
remainder_mstl = mstl_result.model_['remainder']

# Creating a plot using the decomposition_plot function
fig = decomposition_plot(ts_df_mstl.index, observed_mstl, seasonal_mstl, trend_mstl, remainder_mstl)

# Correctly setting the x-axis range for a Plotly figure
fig.update_xaxes(range=["2012-01-01", "2012-02-01"])

fig.show()